In [ ]:
# Ячейка 0 — что делать дальше (прочитайте вывод после Run)

from IPython.display import Markdown, display

display(
    Markdown(
        """
### Yandex Vision OCR в Colab

1. **Runtime → CPU** достаточно (инференс в облаке Яндекса).
2. Выполните ячейки **1 → 4** по порядку.
3. **Креды:** в ячейке 3 задаются `YANDEX_GPT_FOLDER_ID` и `YANDEX_GPT_API_KEY` (или через Colab Secrets — см. комментарий там). Не коммитьте ключи в репозиторий.

**API:** [Vision OCR — начало работы](https://aistudio.yandex.ru/docs/ru/vision/quickstart.html), [концепции OCR](https://aistudio.yandex.ru/docs/ru/vision/concepts/ocr/). Метод `recognizeText`, URL по умолчанию `https://ocr.api.cloud.yandex.net/ocr/v1/recognizeText` (не путать с `YANDEX_GPT_BASE_URL` для LLM).

**Результаты:** `output/yandex_vision/` — `hypotheses/yandex_vision/*.txt`, `yandex_vision_runs.jsonl`, `yandex_vision_summaries.json` (ключ **`yandex_vision`**, те же поля метрик, что у MinerU/Paddle: CER, Final Score, WER в `_diagnostics`).

**Связанные ноутбуки:** MinerU — [`mineru_colab.ipynb`](mineru_colab.ipynb), Paddle — [`paddle_colab.ipynb`](paddle_colab.ipynb), GOT — [`got_colab.ipynb`](got_colab.ipynb).

Ячейка **1** делает `git pull`; при конфликте с `output/*_benchmark` каталоги удаляются и pull повторяется. Ячейка **2** при необходимости скачает `yandex_vision_image_benchmark.py` с Raw (**`OCR_ANALYZE_RAW_BASE`**).
"""
    )
)
print("Готово: выполняйте ячейку 1.")


In [ ]:
# Ячейка 1 — клон репозитория (Colab) + jiwer

from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую репозиторий…")
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочая папка:", Path.cwd().resolve())
    if (REPO_DIR / ".git").is_dir():

        def _pull() -> subprocess.CompletedProcess[str]:
            return subprocess.run(
                ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                capture_output=True,
                text=True,
            )

        pr = _pull()
        combined = (pr.stderr or "") + (pr.stdout or "")
        if pr.returncode != 0 and "would be overwritten by merge" in combined:
            for sub in (
                "mineru_benchmark",
                "paddle_benchmark",
                "got_benchmark",
                "yandex_vision_benchmark",
            ):
                d = REPO_DIR / "output" / sub
                if d.is_dir():
                    print("Удаляю (мешало git pull):", d)
                    shutil.rmtree(d, ignore_errors=True)
            pr = _pull()
            combined = (pr.stderr or "") + (pr.stdout or "")
        if pr.returncode == 0:
            print("git pull: OK")
        else:
            print("git pull: код", pr.returncode)
            print(combined[:1200] if combined else "(нет вывода)")
else:
    print("Не Colab — откройте ноутбук из корня репозитория. cwd:", Path.cwd().resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jiwer"])
print("OK: jiwer. Следующая — ячейка 2.")


In [ ]:
# Ячейка 2 — пути, при необходимости скачать yandex_vision_image_benchmark.py

from __future__ import annotations

import os
import urllib.error
import urllib.request
from pathlib import Path

REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "scripts").is_dir():
            return p
    cwd = Path.cwd().resolve()
    for start in [cwd, *cwd.parents]:
        if (start / "scripts" / "yandex_vision_image_benchmark.py").is_file():
            return start
    return cwd


REPO_ROOT = find_repo_root()
INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
SCRIPT = REPO_ROOT / "scripts" / "yandex_vision_image_benchmark.py"
OUT_DIR = REPO_ROOT / "output" / "yandex_vision"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT.resolve())
print("Скрипт:", SCRIPT.is_file(), SCRIPT)

if not SCRIPT.is_file():
    raw_base = os.environ.get(
        "OCR_ANALYZE_RAW_BASE",
        "https://raw.githubusercontent.com/developer-mixa/OCR-Analyze/main",
    ).rstrip("/")
    url = f"{raw_base}/scripts/yandex_vision_image_benchmark.py"
    print("Скачиваю с Raw:\n ", url)
    SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, SCRIPT)
    except urllib.error.HTTPError as e:
        raise RuntimeError(
            f"HTTP {e.code} при скачивании. Запушьте scripts/yandex_vision_image_benchmark.py или задайте OCR_ANALYZE_RAW_BASE."
        ) from e
    print("Скрипт скачан, байт:", SCRIPT.stat().st_size)

print("Входные PNG:", INPUT_DIR.resolve(), "— есть:", INPUT_DIR.is_dir())
print("Результаты:", OUT_DIR.resolve())
_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print("Найдено PNG:", len(_png))
for p in _png:
    stem = p.stem
    refs = [n for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md") if (INPUT_DIR / n).is_file()]
    print(" ", p.name, "| эталон:", ", ".join(refs) if refs else "нет")
if not _png:
    print("Добавьте PNG в input/data/1.")
print("Следующая — ячейка 3 (креды и model), затем 4.")


In [ ]:
# Ячейка 3 — креды Yandex Cloud + параметры Vision OCR

import os

# Вариант A: Colab Secrets (рекомендуется): создайте секреты с теми же именами.
try:
    from google.colab import userdata  # type: ignore[import-not-found]

    for key in ("YANDEX_GPT_FOLDER_ID", "YANDEX_GPT_API_KEY"):
        try:
            val = userdata.get(key)
            if val and key not in os.environ:
                os.environ[key] = val
                print("Из Colab Secrets:", key, "(длина)", len(val))
        except userdata.SecretNotFoundError:
            pass
except ImportError:
    print("Не Colab — используйте .env или export в оболочке.")

# Вариант B: вставьте сюда значения только для локального теста (не сохраняйте в git):
# os.environ["YANDEX_GPT_FOLDER_ID"] = "b1g..."
# os.environ["YANDEX_GPT_API_KEY"] = "AQVN..."

# Модель Vision OCR: page | table | markdown | handwritten | …
YANDEX_OCR_MODEL = "page"
# Языки: "ru,en" или "*" (авто)
YANDEX_OCR_LANGUAGE_CODES = "ru,en"

print("YANDEX_OCR_MODEL =", YANDEX_OCR_MODEL)
print("YANDEX_OCR_LANGUAGE_CODES =", YANDEX_OCR_LANGUAGE_CODES)
print("folder_id задан:", bool(os.environ.get("YANDEX_GPT_FOLDER_ID") or os.environ.get("YANDEX_OCR_FOLDER_ID")))
print("api_key задан:", bool(os.environ.get("YANDEX_GPT_API_KEY") or os.environ.get("YANDEX_OCR_API_KEY")))
print("IAM задан:", bool(os.environ.get("YANDEX_OCR_IAM_TOKEN") or os.environ.get("YC_IAM_TOKEN")))
print("После правок снова выполните ячейку 4.")


In [ ]:
# Ячейка 4 — прогон Yandex Vision по всем PNG (сетевые запросы в Яндекс)

import json
import os
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "SCRIPT" not in globals():
    raise RuntimeError("Сначала ячейка 2.")
if not SCRIPT.is_file():
    raise RuntimeError("Нет scripts/yandex_vision_image_benchmark.py — см. ячейку 2.")

model = globals().get("YANDEX_OCR_MODEL", "page")
langs = globals().get("YANDEX_OCR_LANGUAGE_CODES", "ru,en")

argv = [
    sys.executable,
    str(SCRIPT),
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUT_DIR),
    "--model",
    str(model),
    "--language-codes",
    str(langs),
]

print("Запуск Yandex Vision OCR, model =", model, ", languages =", langs)
print("Пишем в", OUT_DIR)
subprocess.check_call(argv, cwd=str(REPO_ROOT), env=os.environ.copy())

hyp_dir = OUT_DIR / "hypotheses" / "yandex_vision"
print("\nГипотезы (.txt):")
for p in sorted(hyp_dir.glob("*.txt")):
    print(" ", p.name, p.stat().st_size, "байт")
if not list(hyp_dir.glob("*.txt")):
    print("  нет .txt — см. yandex_vision_runs.jsonl (поле error)")

for name in ("yandex_vision_hypotheses_raw.json", "yandex_vision_hypotheses_concat.txt"):
    fp = OUT_DIR / name
    if fp.is_file():
        print("Доп. файл:", fp.name, fp.stat().st_size, "байт")

summ = OUT_DIR / "yandex_vision_summaries.json"
if summ.is_file():
    print("\n---", summ.name, "---")
    txt = summ.read_text(encoding="utf-8")
    print(txt)
    try:
        outs = json.loads(txt).get("yandex_vision", {}).get("_outputs")
        if outs:
            print("\n_outputs:", json.dumps(outs, ensure_ascii=False, indent=2))
    except json.JSONDecodeError:
        pass
